# Lesson 2: Your First Model  - The ML Process

In Lesson 1, we saw what ML can do. This lesson is about how you actually do it.

Every ML project  - whether it's classifying pet photos, detecting fraud, or generating text  - follows the same basic process:

```
Understand → Prepare → Train → Evaluate → Iterate → Ship
```

We'll work through that process step by step, training a real image classifier along the way. By the end, you'll have a model that recognizes 37 pet breeds, running in a browser.

You won't understand every detail yet. Terms like "loss" and "epoch" will appear before we've explained the math behind them. That's fine  - the goal is to see the full loop once, start recognizing the vocabulary, and build intuition that later lessons will deepen.

In [ ]:
# Setup - run this first
try:
    import google.colab
    !pip install -q fastai
except ImportError:
    pass

In [ ]:
# Plotting helpers (hidden to reduce clutter)
import matplotlib.pyplot as plt
import numpy as np

def plot_class_distribution(labels, title="Class Distribution", figsize=(14, 5)):
    from collections import Counter
    counts = Counter(labels)
    sorted_items = sorted(counts.items(), key=lambda x: x[1], reverse=True)
    breeds, freqs = zip(*sorted_items)
    fig, ax = plt.subplots(figsize=figsize)
    ax.bar(range(len(breeds)), freqs, color='steelblue', alpha=0.7)
    ax.set_xticks(range(len(breeds)))
    ax.set_xticklabels([b.replace('_', ' ').title() for b in breeds], rotation=45, ha='right', fontsize=8)
    ax.set_ylabel('Number of Images')
    ax.set_title(title)
    ax.axhline(y=np.mean(freqs), color='red', linestyle='--', label=f'Mean: {np.mean(freqs):.0f}')
    ax.legend()
    plt.tight_layout()

def plot_overfit_underfit_demo(figsize=(14, 4)):
    fig, axes = plt.subplots(1, 3, figsize=figsize)
    epochs = range(1, 11)

    ax = axes[0]
    ax.plot(epochs, [2.5 - i*0.05 for i in epochs], 'b-', label='Train Loss', linewidth=2)
    ax.plot(epochs, [2.6 - i*0.04 for i in epochs], 'r-', label='Valid Loss', linewidth=2)
    ax.set_title('Underfitting\n(Model too simple)', fontsize=12, fontweight='bold')
    ax.set_xlabel('Epoch'); ax.set_ylabel('Loss'); ax.legend(); ax.set_ylim(0, 3)
    ax.text(5, 2.7, 'Both losses stay high!', ha='center', fontsize=10, color='red')

    ax = axes[1]
    ax.plot(epochs, [2.5*np.exp(-0.4*i)+0.2 for i in epochs], 'b-', label='Train Loss', linewidth=2)
    ax.plot(epochs, [2.7*np.exp(-0.35*i)+0.3 for i in epochs], 'r-', label='Valid Loss', linewidth=2)
    ax.set_title('Good Fit\n(Model generalizes well)', fontsize=12, fontweight='bold', color='green')
    ax.set_xlabel('Epoch'); ax.set_ylabel('Loss'); ax.legend(); ax.set_ylim(0, 3)
    ax.text(5, 2.7, 'Both losses decrease together', ha='center', fontsize=10, color='green')

    ax = axes[2]
    ax.plot(epochs, [2.5*np.exp(-0.5*i)+0.1 for i in epochs], 'b-', label='Train Loss', linewidth=2)
    ax.plot(epochs, [2.5*np.exp(-0.3*i)+0.2+0.05*(i-3)**2*(i>3) for i in epochs], 'r-', label='Valid Loss', linewidth=2)
    ax.axvline(x=4, color='orange', linestyle='--', alpha=0.7)
    ax.set_title('Overfitting\n(Model memorizes training data)', fontsize=12, fontweight='bold', color='red')
    ax.set_xlabel('Epoch'); ax.set_ylabel('Loss'); ax.legend(); ax.set_ylim(0, 3)
    ax.text(7, 2.5, 'Valid loss goes UP!', ha='center', fontsize=10, color='red')
    ax.text(4.2, 0.3, 'stop here', fontsize=9, rotation=90, color='orange')
    plt.tight_layout()

def plot_learning_rate_demo(figsize=(14, 4)):
    fig, axes = plt.subplots(1, 3, figsize=figsize)
    epochs = range(1, 21)

    axes[0].plot(epochs, [3.0 - 0.05*i for i in epochs], 'b-', linewidth=2)
    axes[0].set_title('Learning Rate TOO LOW', fontsize=12, fontweight='bold')
    axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss'); axes[0].set_ylim(0, 4)
    axes[0].text(10, 3.5, 'Very slow progress', ha='center', fontsize=10, color='blue')

    axes[1].plot(epochs, [3.0*np.exp(-0.3*i)+0.3 for i in epochs], 'g-', linewidth=2)
    axes[1].set_title('Learning Rate JUST RIGHT', fontsize=12, fontweight='bold', color='green')
    axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Loss'); axes[1].set_ylim(0, 4)
    axes[1].text(10, 3.5, 'Steady improvement', ha='center', fontsize=10, color='green')

    np.random.seed(42)
    axes[2].plot(epochs, [2.5 + np.random.randn()*0.5 + 0.1*i for i in epochs], 'r-', linewidth=2)
    axes[2].set_title('Learning Rate TOO HIGH', fontsize=12, fontweight='bold', color='red')
    axes[2].set_xlabel('Epoch'); axes[2].set_ylabel('Loss'); axes[2].set_ylim(0, 6)
    axes[2].text(10, 5.5, 'Erratic, may diverge!', ha='center', fontsize=10, color='red')
    plt.tight_layout()

def show_sample_images(images_with_labels, ncols=4, figsize=(14, 10)):
    n = len(images_with_labels)
    rows = (n + ncols - 1) // ncols
    fig, axes = plt.subplots(rows, ncols, figsize=figsize)
    axes = axes.flatten()
    for ax, (img, label) in zip(axes, images_with_labels):
        ax.imshow(img)
        ax.set_title(label.replace('_', ' ').title(), fontsize=10)
        ax.axis('off')
    for ax in axes[n:]:
        ax.axis('off')
    plt.suptitle('Random Samples from the Dataset', fontsize=14, y=1.01)
    plt.tight_layout()

def plot_training_curves(train_losses, val_accs, figsize=(14, 5)):
    fig, axes = plt.subplots(1, 2, figsize=figsize)
    axes[0].plot(train_losses, alpha=0.7)
    axes[0].set_xlabel('Batch'); axes[0].set_ylabel('Loss')
    axes[0].set_title('Training Loss Over Time')
    axes[0].grid(True, alpha=0.3)
    epochs = range(1, len(val_accs) + 1)
    axes[1].plot(epochs, val_accs, 'g-o', linewidth=2, markersize=8)
    axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Accuracy')
    axes[1].set_title('Validation Accuracy Per Epoch')
    axes[1].grid(True, alpha=0.3); axes[1].set_ylim(0, 1)
    plt.tight_layout()

def plot_prediction_with_probs(img, actual, predicted, top_breeds, top_probs, figsize=(14, 6)):
    fig, axes = plt.subplots(1, 2, figsize=figsize)
    axes[0].imshow(img)
    axes[0].axis('off')
    correct = "Correct!" if predicted == actual else "Wrong"
    color = "green" if predicted == actual else "red"
    axes[0].set_title(f"Actual: {actual.replace('_', ' ').title()}\n"
                      f"Predicted: {predicted.replace('_', ' ').title()}\n"
                      f"{correct}", fontsize=12, color=color)
    colors = ['green' if b.lower().replace(' ', '_') == actual.lower() else 'steelblue' for b in top_breeds]
    axes[1].barh(range(len(top_breeds)-1, -1, -1), top_probs, color=colors, alpha=0.7)
    axes[1].set_yticks(range(len(top_breeds)-1, -1, -1))
    axes[1].set_yticklabels(top_breeds)
    axes[1].set_xlabel('Probability')
    axes[1].set_title('Top 5 Predictions')
    axes[1].set_xlim(0, 1)
    plt.tight_layout()

def plot_prediction_grid(predictions, figsize=(14, 10)):
    n = len(predictions)
    cols = min(3, n)
    rows = (n + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=figsize)
    axes = np.array(axes).flatten()
    for ax, (img, actual, predicted, confidence) in zip(axes, predictions):
        ax.imshow(img)
        ax.axis('off')
        correct = predicted == actual
        color = 'green' if correct else 'red'
        symbol = 'Y' if correct else 'N'
        ax.set_title(f"Actual: {actual.replace('_', ' ').title()}\n"
                     f"Predicted: {predicted.replace('_', ' ').title()}\n"
                     f"Confidence: {confidence:.1%} {symbol}",
                     fontsize=10, color=color)
    for ax in axes[n:]:
        ax.axis('off')
    plt.suptitle('Model Predictions on Random Images', fontsize=14, y=1.02)
    plt.tight_layout()

print("Plotting helpers loaded!")

In [ ]:
import warnings
warnings.filterwarnings('ignore')

from fastai.vision.all import *
import fastai
import random
import torch

random.seed(42)

# Check for GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Libraries loaded! fastai version: {fastai.__version__}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("No GPU detected - training will use CPU (slower but works fine)")

## Step 1: Understand Your Data

Before training anything, understand what you're working with. What does the data look like? How is it organized? How many classes? Is it balanced? Are some classes similar to each other? Every ML project starts here - and skipping this step is how you waste hours training on bad data.

We'll use the **Oxford-IIIT Pet Dataset**: photos of cats and dogs organized by breed. Our task is to classify each photo into one of 37 breeds. That's a lot harder than it sounds - some breeds look nearly identical, and images vary wildly in angle, lighting, and background.

In [ ]:
# Download and point to the dataset
path = untar_data(URLs.PETS)
image_path = path / 'images'

# List all images
image_files = list(image_path.glob('*.jpg'))

# Check for corrupted images (always do this with downloaded data)
failed = verify_images(image_files)
if len(failed) > 0:
    print(f"Found {len(failed)} corrupted images - removing them")
    failed.map(Path.unlink)
    image_files = list(image_path.glob('*.jpg'))

# Extract breed from filename: "beagle_123.jpg" -> "beagle"
def get_breed(filename):
    parts = filename.stem.split('_')
    return '_'.join(parts[:-1])

all_breeds = [get_breed(f) for f in image_files]
unique_breeds = set(all_breeds)

print(f"Dataset location: {image_path}")
print(f"Total images:     {len(image_files):,}")
print(f"Number of breeds: {len(unique_breeds)}")
print(f"Images per breed: ~{len(image_files) // len(unique_breeds)}")

The breed is encoded in the filename  - `beagle_123.jpg` means breed "beagle", image 123. Let's look at a few:

In [ ]:
for f in image_files[:8]:
    print(f"  {f.name:40s} → {get_breed(f)}")

#### Why is this a hard problem?

37 breeds might not sound like much, but think about what the model has to deal with:
- **Random guessing** would only be right 2.7% of the time (1/37)
- **Within-class variation**: the same breed can look very different depending on age, angle, lighting, and background
- **Between-class similarity**: some breeds look nearly identical to humans, let alone a computer

Let's look at some breed pairs that will be challenging:

In [ ]:
# Show similar-looking breed pairs side by side
similar_pairs = [
    ('Ragdoll', 'Birman'),
    ('american_pit_bull_terrier', 'staffordshire_bull_terrier'),
    ('Bengal', 'Egyptian_Mau'),
]

fig, axes = plt.subplots(len(similar_pairs), 4, figsize=(16, 4 * len(similar_pairs)))
for row, (breed_a, breed_b) in enumerate(similar_pairs):
    imgs_a = [f for f in image_files if get_breed(f).lower() == breed_a.lower()]
    imgs_b = [f for f in image_files if get_breed(f).lower() == breed_b.lower()]
    for col, img_path in enumerate(random.sample(imgs_a, 2) + random.sample(imgs_b, 2)):
        ax = axes[row][col]
        ax.imshow(PILImage.create(img_path))
        breed = get_breed(img_path).replace('_', ' ').title()
        ax.set_title(breed, fontsize=10, fontweight='bold',
                    color='#e74c3c' if col < 2 else '#3498db')
        ax.axis('off')
plt.suptitle('Can you tell these apart? Your model will need to.', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# Visualize the class distribution
plot_class_distribution(all_breeds, title="Number of Images per Breed")
plt.show()

print("→ The dataset is fairly balanced  - each breed has roughly 200 images.")
print("  Some of these breeds look very similar. We'll see later which ones the model struggles to tell apart.")

Always look at your data before doing anything else. Numbers tell you the shape  - images tell you the challenge.

In [ ]:
# Show random samples from the dataset
samples = [(PILImage.create(f), get_breed(f)) for f in random.sample(image_files, 12)]
show_sample_images(samples)
plt.show()

#### What does the model actually see?

To us, that's a photo of a beagle. To the model, it's a grid of numbers. Each pixel has 3 values (red, green, blue), each between 0 and 255. A 224x224 image becomes 224 x 224 x 3 = **150,528 numbers**.

In Lesson 1's regression demo, the model worked with just 2 numbers (income and house price). Here, each single image is 150,528 numbers. The model needs to find patterns in those numbers that distinguish "beagle" from "poodle".

In [ ]:
# What does the model actually see?
sample_img = PILImage.create(random.choice(image_files))
img_tensor = tensor(sample_img.resize((224, 224))).permute(2, 0, 1).float() / 255.0

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].imshow(sample_img.resize((224, 224)))
axes[0].set_title('What we see')
axes[0].axis('off')

patch = img_tensor[0, 50:58, 50:58]
axes[1].imshow(patch.numpy(), cmap='Reds', vmin=0, vmax=1)
for i in range(8):
    for j in range(8):
        axes[1].text(j, i, f'{patch[i,j]:.2f}', ha='center', va='center', fontsize=7,
                    color='white' if patch[i,j] > 0.5 else 'black')
axes[1].set_title('Zoomed in: 8x8 patch (red channel)\nEach cell is one pixel value')
axes[1].set_xticks([]); axes[1].set_yticks([])
plt.tight_layout()
plt.show()

print(f"Image shape: {img_tensor.shape} = 3 channels x 224 x 224 pixels = {img_tensor.numel():,} numbers")

## Step 2: Prepare the Data

Now we need to get this data into a format the model can learn from. That means: splitting into training and validation sets, resizing images to a consistent size, and adding augmentation. In later lessons you'll build these pipelines manually with PyTorch  - here fastai's DataBlock handles it.

### Training vs. validation split

Think of studying for an exam: if you only practice with the actual exam questions, you'd ace it  - but that doesn't mean you learned the material. You need to test yourself on questions you haven't seen.

Same idea here. We hold back 20% of images as a **validation set**  - the model never sees these during training. Performance on the validation set tells us if the model actually learned to recognize breeds, or just memorized the training images.

If you've used scikit-learn, you'd normally call `train_test_split()`. fastai's `RandomSplitter` does the same thing, but integrated with the rest of the pipeline  - splitting, augmentation, and batching all stay in sync.

### The DataBlock

fastai's DataBlock lets us configure the entire data pipeline in one place. Think of it as answering a series of questions about your data:

In [ ]:
# Create the DataBlock - each line answers a question about our data
pets = DataBlock(
    blocks=(ImageBlock, CategoryBlock),            # What type is the input/output? Images in, categories out.
    get_items=get_image_files,                      # How do we find the data? Scan folder for image files.
    splitter=RandomSplitter(valid_pct=0.2, seed=42),# How to split? 80% train, 20% validation, reproducible.
    get_y=get_breed,                                # Where are the labels? Extract breed from filename.
    item_tfms=Resize(460),                          # First resize? Make all images 460px (before batching).
    batch_tfms=aug_transforms(size=224)             # Then what? Augment + crop to 224px (on GPU, per batch).
)

print("DataBlock created!")

#### Why resize to 460 first, then 224?

This is a clever trick. If you resize directly to 224, you lose detail. And when augmentation applies random rotations or warps to a 224px image, empty pixels appear at the edges.

The solution: resize to 460 first (preserving more detail), then let augmentation randomly crop/warp down to 224. This means the model sees different 224-pixel regions of each larger image, effectively increasing dataset diversity while maintaining image quality.

```
Original image (any size)  ->  Resize(460px)  ->  aug_transforms(224px)  ->  Model
                                 (preserve detail)   (random crop + augment)
```

In [ ]:
# Create the actual DataLoaders
dls = pets.dataloaders(image_path, bs=64)

print(f"Training:   {len(dls.train.dataset)} images in {len(dls.train)} batches")
print(f"Validation: {len(dls.valid.dataset)} images in {len(dls.valid)} batches")
print(f"Classes:    {len(dls.vocab)} breeds")

### Verify: does the data look right?

Always check your data pipeline before training. If labels are wrong or images are corrupted, you'll waste time training on garbage.

In [ ]:
# Show a batch from the training set
dls.show_batch(max_n=12, figsize=(14, 10))

Notice some images are flipped, slightly rotated, or have different lighting. That's **data augmentation**  - we randomly transform each training image so the model never sees the exact same image twice. A beagle is still a beagle whether it's facing left or right. This forces the model to learn features that actually matter, not memorize specific photos.

We only augment the training data. The validation set stays unchanged so we can fairly test the model.

## Step 3: Train the Model

Data is ready. Now we pick a model and train it. This is where machine learning actually happens: the model looks at images, makes predictions, checks how wrong it was, and adjusts. Repeat thousands of times.

In Lesson 1's regression demo, we saw this process with just 2 numbers - a weight and a bias defining a straight line. The model started with random values, measured the loss, and used gradient descent to improve. The result was a line that fit the housing data.

Here it's the exact same process, just at a wildly different scale. Instead of 2 numbers defining a line, we have about 21 million numbers defining how to interpret images. Same engine: start random, measure loss, adjust with gradient descent.

### Transfer learning: don't start from scratch

If you've never seen a Bengal cat before, but you know what cats look like in general - eyes, ears, fur, body shapes - learning to recognize Bengals isn't that hard. You just need to learn the specific coat pattern.

Same idea here. Instead of training from scratch (which would need millions of images), we start with **ResNet34** - a model already trained on 1.2 million images from ImageNet. Through that training, it learned to detect edges, textures, shapes, and objects. We keep that knowledge and just teach it our 37 pet breeds. This is called **transfer learning**.

```
PRE-TRAINED MODEL (ResNet34)                    OUR MODEL
┌─────────────────────────────────────┐         ┌─────────────────────────────────────┐
│                                     │         │                                     │
│  Input Layer                        │         │  Input Layer (same)                 │
│       ↓                             │         │       ↓                             │
│  Feature Extraction Layers          │  ───►   │  Feature Extraction Layers (KEEP)  │
│  (edges, textures, shapes)          │         │  (already knows how to "see")       │
│       ↓                             │         │       ↓                             │
│  Classification Head                │         │  NEW Classification Head            │
│  (1000 ImageNet classes)            │         │  (37 pet breeds)                    │
│                                     │         │                                     │
└─────────────────────────────────────┘         └─────────────────────────────────────┘
```

We keep the learned feature layers (they already work great) and replace only the final classification head. Then `fine_tune()` trains everything together so the model adapts its knowledge to pet breeds specifically.

In [ ]:
# Create a learner: our data + a pre-trained ResNet34
learn = vision_learner(dls, resnet34, metrics=accuracy)

total_params = sum(p.numel() for p in learn.model.parameters())
trainable_params = sum(p.numel() for p in learn.model.parameters() if p.requires_grad)

print(f"Model: ResNet34 (pre-trained on ImageNet)")
print(f"Task:  Classify {len(dls.vocab)} pet breeds")
print(f"")
print(f"Total parameters:     {total_params:>12,}")
print(f"Trainable parameters: {trainable_params:>12,}")
print(f"")
print(f"For comparison: L1's regression model had 2 parameters.")
print(f"Same gradient descent process, {total_params // 2:,}x more numbers to optimize.")

In [ ]:
# Train! Watch the loss decrease and accuracy increase.
learn.fine_tune(3)

### Reading the output

That table is your dashboard for the training process. Here's what each column means:

- **epoch** - one complete pass through all training images
- **train_loss** - how wrong the model was on training data (lower is better)
- **valid_loss** - how wrong on validation images (the real test - data it hasn't trained on)
- **accuracy** - what percentage of validation images it classified correctly

`fine_tune(3)` works in two phases:
- **Row 0 (epoch 0):** Only the new classification head trains. The feature extraction layers are frozen - the model already knows how to "see", it just needs to learn our 37 breeds.
- **Rows 1-3 (epochs 1-3):** The entire network unfreezes and fine-tunes. Now the earlier layers can adapt their edge/texture detection to work better for pet breeds specifically.

**Loss vs accuracy - what's the difference?** Both appear in the output, but they serve different purposes. The **loss** is what the model actually optimizes - it's the number gradient descent tries to minimize. **Accuracy** is for us humans - it's the metric we care about. The model doesn't directly optimize accuracy (the math doesn't work that way), it optimizes loss, and accuracy improves as a side effect. You'll see why this distinction matters when we build our own loss functions in L3.

A few things to notice: both training loss and validation loss went down. That's good - the model is learning AND generalizing. If validation loss started going UP while training loss kept falling, the model would be memorizing instead of learning. That's called **overfitting**, and we'll explore it properly in Lesson 3.

There's also a number controlling how much the model adjusts on each step - the **learning rate**. fastai picks a good one automatically. Understanding why that number matters and how to choose it is also Lesson 3 territory.

Let's immediately see what the model's predictions actually look like:

In [ ]:
# Quick visual check - how do predictions look on validation images?
learn.show_results(max_n=8, figsize=(12, 10))

Each image shows the actual breed (label) and what the model predicted. Most should match - but look for any mismatches. Those are the interesting cases we'll dig into during the evaluation step.

In [ ]:
# Visualize the training progress and save data for later
train_losses = [x.item() for x in learn.recorder.losses]
val_accs = [float(x[-1]) for x in learn.recorder.values]
val_losses = [float(x[-2]) for x in learn.recorder.values]
v1_final_acc = val_accs[-1]

plot_training_curves(train_losses, val_accs)
plt.show()

print(f"Final validation accuracy: {v1_final_acc:.1%}")

The loss curve tells the same story as the table, just visually. Loss drops sharply at first (the model is learning the easy patterns), then levels off. In later lessons you'll learn to read these curves to diagnose training problems.

### What does "loss" actually mean?

Loss is a single number measuring **how wrong** the model's predictions are  - a "wrongness score." High loss means predictions are far off, low loss means they're close to correct.

During training, the model tries to minimize this loss. Each time it sees a batch of images, it makes predictions, compares them to the correct labels, calculates the loss, and adjusts its weights to reduce it. The exact formula (called **cross-entropy loss** for classification) is something we'll derive from scratch in Lesson 3. For now, just remember: lower loss = better model.

### Two things to watch for: overfitting and underfitting

These are two of the most common problems in machine learning. You don't need to fully understand them yet, but you should know what they look like.

**Underfitting** means the model is too simple to capture the patterns. Both training and validation loss stay high. The fix is usually a more complex model or more training.

**Overfitting** means the model memorized the training data instead of learning general patterns. Training loss is low, but validation loss is high (or starts going up). It's like memorizing exam answers instead of understanding the material.

We'll explore both in depth in Lesson 3. For now, here's what they look like on a loss curve:

In [ ]:
# Visualize what underfitting, good fit, and overfitting look like
plot_overfit_underfit_demo()
plt.show()

print("→ Our model looks healthy! Both losses decreased and stayed close together.")
print("  That puts us in the 'good fit' zone.")

Now let's look at our model's actual training curves and compare them to those patterns:

In [ ]:
# Plot our model's actual training curves (using data saved earlier)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(train_losses, alpha=0.5, color='steelblue')
axes[0].set_xlabel('Batch')
axes[0].set_ylabel('Training Loss')
axes[0].set_title('Our Model: Training Loss Per Batch')
axes[0].grid(True, alpha=0.3)

epochs = range(1, len(val_accs) + 1)
ax2 = axes[1].twinx()
axes[1].plot(epochs, val_losses, 'r-o', label='Valid Loss', linewidth=2)
ax2.plot(epochs, val_accs, 'g-s', label='Accuracy', linewidth=2)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Valid Loss', color='red')
ax2.set_ylabel('Accuracy', color='green')
axes[1].set_title('Our Model: Validation Loss & Accuracy')
lines1, labels1 = axes[1].get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
axes[1].legend(lines1 + lines2, labels1 + labels2)

plt.tight_layout()
plt.show()

print(f"Final valid loss: {val_losses[-1]:.3f}")
print(f"Final accuracy:   {val_accs[-1]:.1%}")

### The learning rate: how fast should the model learn?

The **learning rate** controls how much the model adjusts its weights after each batch. Think of searching for the lowest point in a valley while blindfolded  - the learning rate is your step size.

- **Too small**: You'll get there eventually, but it takes forever
- **Just right**: Steady progress toward the bottom
- **Too large**: You overshoot and bounce around, maybe climbing higher

fastai picks a good learning rate automatically, but understanding why it matters will be important when you tune models yourself. We'll dig into the math in Lesson 3.

In [ ]:
# Visualize the effect of different learning rates
plot_learning_rate_demo()
plt.show()

## Step 4: Evaluate

The model says ~90% accuracy. But where does it fail, and why? Understanding the mistakes is often more valuable than the accuracy number. This step is where you build real intuition about your model.

In [ ]:
# Create an interpretation object
interp = ClassificationInterpretation.from_learner(learn)

print("Interpretation object created!")
print("This analyzes the model's predictions on the validation set.")

### Which breeds does it confuse?

A **confusion matrix** shows, for each true class, what the model predicted. With 37 breeds the full matrix is hard to read, so let's look at the most confused pairs:

In [ ]:
# Show the most confused pairs
print("Most confused breed pairs:\n")

confused = interp.most_confused(min_val=2)
for true_breed, pred_breed, count in confused[:10]:
    true_name = true_breed.replace('_', ' ').title()
    pred_name = pred_breed.replace('_', ' ').title()
    print(f"  {true_name:30s} → {pred_name:30s} ({count})")

The confusions make sense  - Ragdoll vs Birman, Pit Bull vs Staffordshire Terrier, Bengal vs Egyptian Mau. These are breeds that even humans find hard to tell apart. The model's mistakes are between genuinely similar-looking animals, which means it learned real visual features, not just random patterns.

### The biggest mistakes

Let's look at images where the model was most confidently wrong:

In [ ]:
# Show images with highest loss (biggest mistakes)
interp.plot_top_losses(9, figsize=(15, 12))

You'll often find genuinely ambiguous images, unusual angles, or even mislabeled data in the dataset itself. Looking at errors tells you whether the model needs improvement or the data does.

### Making predictions

Let's use the trained model on specific images to see predictions and confidence:

In [ ]:
# Pick a random image and predict
test_img_path = random.choice(image_files)
test_img = PILImage.create(test_img_path)
pred_class, pred_idx, probs = learn.predict(test_img)
actual_breed = get_breed(test_img_path)

top5_idx = probs.argsort(descending=True)[:5]
top5_breeds = [dls.vocab[i].replace('_', ' ').title() for i in top5_idx]
top5_probs = [probs[i].item() for i in top5_idx]

plot_prediction_with_probs(test_img, actual_breed, pred_class, top5_breeds, top5_probs)
plt.show()

In [ ]:
# Quick batch of predictions
predictions = []
for img_path in random.sample(image_files, 6):
    img = PILImage.create(img_path)
    pred_class, pred_idx, probs = learn.predict(img)
    actual = get_breed(img_path)
    predictions.append((img, actual, pred_class, probs[pred_idx].item()))

plot_prediction_grid(predictions)
plt.show()

## Step 5: Iterate

You've trained one model and evaluated it. In practice, the first model is rarely the best. The iterate step is where you form a hypothesis based on your evaluation, change one thing, and see if it helps.

Here are the knobs you can turn:

| What to change | Example | When it might help |
|----------------|---------|-------------------|
| More epochs | `fine_tune(5)` instead of 3 | Accuracy was still climbing |
| Bigger architecture | `resnet50` instead of `resnet34` | Model seems to plateau early |
| Larger image size | `Resize(320)` | Fine details matter (similar breeds) |
| Different augmentation | More aggressive transforms | Model overfits |

The key habit: **change one thing at a time**. If you change the architecture AND the epochs AND the image size, you won't know which change helped.

Let's try one change. Our accuracy was still climbing at epoch 3  - let's give it 5 epochs instead:

In [ ]:
# Same setup, more epochs
learn_v2 = vision_learner(dls, resnet34, metrics=accuracy)
learn_v2.fine_tune(5)

Let's compare the two models explicitly:

In [ ]:
# Compare v1 (3 epochs) vs v2 (5 epochs)
v2_final_acc = float(learn_v2.recorder.values[-1][-1])  # last metric = accuracy

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar(['v1 (3 epochs)', 'v2 (5 epochs)'], [v1_final_acc, v2_final_acc],
              color=['#3498db', '#2ecc71'], width=0.5)
ax.set_ylabel('Validation Accuracy')
ax.set_title('Accuracy Comparison: More Epochs')
ax.set_ylim(min(v1_final_acc, v2_final_acc) - 0.05, 1.0)
for bar, acc in zip(bars, [v1_final_acc, v2_final_acc]):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
            f'{acc:.1%}', ha='center', fontweight='bold')
plt.tight_layout()
plt.show()

diff = v2_final_acc - v1_final_acc
print(f"v1 accuracy: {v1_final_acc:.1%}")
print(f"v2 accuracy: {v2_final_acc:.1%}")
print(f"Difference:  {diff:+.1%}")

This is the iterate mindset: form a hypothesis ("accuracy was still climbing, more epochs might help"), change one thing, measure the result. Sometimes it helps, sometimes it doesn't. Either way, you learned something.

Other things you could try in the exercise notebook:
- Bigger architecture (ResNet50 instead of ResNet34)
- Larger images (320px instead of 224px)
- More aggressive augmentation

The key habit: **change one thing at a time.** If you change the architecture AND the epochs AND the image size simultaneously, you won't know which change helped.

## Step 6: Ship

The model works. Now save it so it works without this notebook, and optionally deploy it to a browser. This is the last step in the process  - and the one most courses skip entirely.

In [ ]:
import os

model_path = '/tmp/pet_classifier.pkl'
learn.export(model_path)

size_mb = os.path.getsize(model_path) / (1024 * 1024)

print(f"Model saved to: {model_path}")
print(f"File size:      {size_mb:.1f} MB")
print(f"\nFor comparison:")
print(f"  ResNet18 (smaller)     →  ~45 MB")
print(f"  ResNet34 (ours)        →  ~85 MB")
print(f"  ResNet50 (larger)      → ~100 MB")
print(f"  GPT-2 (small LLM)     → ~500 MB")
print(f"  Llama 3 8B (quantized) →   ~4 GB")

### What's inside the .pkl file?

That single file contains everything needed to make predictions without any training data:

- **Model architecture**  - the ResNet34 structure (which layers, how they connect)
- **Trained weights**  - millions of numbers the model learned during training
- **Data transforms**  - how to resize, normalize, and preprocess new images
- **Class vocabulary**  - the mapping from index to breed name (so the model says "beagle" instead of "class 7")

In [ ]:
# Prove it works: load the saved model and make a prediction
loaded_model = load_learner(model_path)

test_img_path = random.choice(image_files)
test_img = PILImage.create(test_img_path)
pred_class, pred_idx, probs = loaded_model.predict(test_img)
actual_breed = get_breed(test_img_path)

top5_idx = probs.argsort(descending=True)[:5]
top5_breeds = [loaded_model.dls.vocab[i].replace('_', ' ').title() for i in top5_idx]
top5_probs = [probs[i].item() for i in top5_idx]

plot_prediction_with_probs(test_img, actual_breed, pred_class, top5_breeds, top5_probs)
plt.show()

print("→ This prediction came from the saved .pkl file, not the training session.")
print("  The model works independently now.")

### Deploy to a browser

That `.pkl` file is all you need to serve predictions. The `classifier_deploy/` folder in this lesson wraps it in a full-stack web app: FastAPI backend, Streamlit frontend, PostgreSQL for logging. Let's walk through the code - the ML part is surprisingly small. The rest is standard web development.

### The Architecture

```
Browser  ->  Streamlit UI (upload image, see result)
                |
                v
             FastAPI (POST /predict)
             |- loads pet_classifier.pkl at startup
             '- logs predictions to PostgreSQL
```

Three containers, each with one job. The API loads the model once at startup and serves predictions. The frontend lets users upload images. The database logs every prediction for monitoring.

### The Prediction Endpoint: predict.py

```python
@router.post("/predict")
def predict(file: UploadFile = File(...), db: Session = Depends(get_db)):
    start = time.perf_counter()

    # THE ML PART - just 3 lines:
    image_bytes = file.file.read()
    img = PILImage.create(io.BytesIO(image_bytes))
    pred, idx, probs = learn.predict(img)

    processing_time_ms = int((time.perf_counter() - start) * 1000)

    # Log to database for monitoring
    db.add(Prediction(
        prediction=str(pred),
        confidence=float(probs[idx]),
        probabilities={learn.dls.vocab[i]: round(float(p), 4)
                       for i, p in enumerate(probs)},
        processing_time_ms=processing_time_ms,
    ))
    db.commit()
    return {"prediction": str(pred), "confidence": round(float(probs[idx]), 4), ...}
```

The ML-specific code is 3 lines: create a PIL image from bytes, call `learn.predict()`, read the probabilities. Everything else is logging, timing, and response formatting. This is typical - the model inference code is always tiny compared to the infrastructure around it.

### The Frontend: Streamlit

```python
st.title("Pet Breed Classifier")
uploaded_file = st.file_uploader("Choose an image", type=["jpg", "jpeg", "png"])

if uploaded_file is not None:
    image = Image.open(uploaded_file)
    st.image(image, caption="Uploaded image", use_container_width=True)

    response = requests.post(
        "http://api:8000/predict",
        files={"file": ("image.jpg", uploaded_file.getvalue(), "image/jpeg")},
    )
    result = response.json()
    st.subheader(f"Prediction: {result['prediction']}")
    st.metric("Confidence", f"{result['confidence']:.1%}")
```

The entire frontend is about 40 lines. Streamlit handles the UI - file upload widget, image display, metrics. It sends the image to the FastAPI backend and displays the result. No JavaScript needed.

### Docker Compose: Putting It Together

```yaml
services:
  api:
    build: { context: ., dockerfile: docker/Dockerfile.api }
    ports: ["8000:8000"]
    volumes: ["./models:/app/models:ro"]    # Model file mounted read-only
    depends_on:
      db: { condition: service_healthy }

  frontend:
    build: { context: ., dockerfile: docker/Dockerfile.frontend }
    ports: ["8501:8501"]
    depends_on: [api]

  db:
    image: postgres:16-alpine
    healthcheck:
      test: ["CMD-SHELL", "pg_isready -U petapp"]
```

Three services. The API waits for PostgreSQL to be healthy before starting. The model file is mounted read-only. The frontend depends on the API being up.

### Running It

```bash
# 1. Copy your trained model into the deploy folder
cp /tmp/pet_classifier.pkl classifier_deploy/models/

# 2. Start everything
cd classifier_deploy
docker-compose up --build

# 3. Open in browser
# Frontend: http://localhost:8501
# API docs: http://localhost:8000/docs
```

You can also test the API directly with curl:

```bash
curl -X POST http://localhost:8000/predict \
  -F "file=@my_cat_photo.jpg"
```

The API uses CPU-only PyTorch to keep the Docker image small. Inference takes ~50-200ms per image - fast enough for interactive use.

There's a deeper deployment lesson later in the course covering cloud deployment (AWS EC2), model optimization, and monitoring. This is just to close the loop: **train a model -> save it -> serve it -> use it from a browser.** One lesson, full cycle.

## The Process

You just completed the ML process:

```
Understand → Prepare → Train → Evaluate → Iterate → Ship
```

This exact loop applies to every ML project in this course and beyond. The tools will change  - in Lesson 6 you'll use PyTorch instead of fastai, in Lesson 9 you'll work with images from scratch, in Lesson 12 you'll deploy to AWS. But the process stays the same.

Some of the steps will look different depending on the problem. Tabular data needs different preparation than images. Regression uses different metrics than classification. But the loop  - understand your data, prepare it, train a model, evaluate the results, iterate, ship  - that's universal.

The **exercise notebook** (`02_exercise_train_your_own.ipynb`) gives you a chance to run this process yourself on a different dataset. That's where it really clicks  - when you have to figure out each step without the walkthrough.

### A word of caution

This model works great on pet photos that look like the training data - clear photos of cats and dogs, similar angles and lighting to what it trained on. But it will confidently give wrong answers on:

- **Out-of-domain data**: Show it a photo of a car and it'll still pick a breed. It has no "I don't know" option - it always picks the most likely breed from its 37 choices
- **Unusual angles or conditions**: A heavily backlit silhouette of a dog, or a close-up of just an ear
- **Breeds it hasn't seen**: If you show it a breed not in the 37, it can't say "unknown"

This isn't a flaw in our model specifically - it's a fundamental property of classifiers. They pick the best option from the options they know. Understanding these limitations is part of being a responsible ML practitioner. We'll talk more about this throughout the course.

## Key Takeaways

### The process

```
Understand → Prepare → Train → Evaluate → Iterate → Ship
```

We followed these steps to build a pet breed classifier with ~90% accuracy. The same steps apply to every supervised ML project  - the data, tools, and models change, but the loop doesn't.

### Vocabulary you encountered

| Term | What it means |
|------|---------------|
| **Epoch** | One complete pass through all training data |
| **Loss** | A number measuring how wrong the model is (lower = better) |
| **Accuracy** | Percentage of correct predictions on the validation set |
| **Training set** | Data the model learns from |
| **Validation set** | Data held back to test if the model generalizes |
| **Transfer learning** | Starting with a model pre-trained on different data |
| **Fine-tuning** | Adapting a pre-trained model to your specific task |
| **Data augmentation** | Creating random variations of training images |
| **Confusion matrix** | Table showing which classes the model confuses |
| **Overfitting** | When the model memorizes training data instead of learning (explored in L3) |
| **Learning rate** | How much the model adjusts on each step (explored in L3) |

Don't worry about memorizing these. They'll keep showing up in every lesson until they're second nature.

### What we glossed over

We used `fine_tune()` without understanding what's inside. A lot happened behind that one line:
- How does the loss function actually work?
- What's inside ResNet34?
- How do gradients update the weights?
- Why do neural networks learn at all?

**That's what the rest of this course is about.** Starting next lesson, we'll build understanding from the ground up  - implementing gradient descent, understanding backpropagation, and eventually building our own neural network from scratch.

## What's Next?

**Exercise:** Run the process yourself with a new dataset. See `02_exercise_train_your_own.ipynb`  - pick a dataset, work through all six steps, and see how the process adapts to different data.

**Lesson 3: Regression and Gradient Descent**

We used the word "loss" a lot without explaining what it actually means. Lesson 3 dives into the math: what loss functions are, how gradient descent works, and how models actually learn. You'll understand what was happening behind `fine_tune()`.

**See you there!**

<div style="text-align: center; color: #888; font-size: 0.85em; margin-top: 40px; padding-top: 10px; border-top: 1px solid #ddd;">
© 2025 Utvecklarakademin UA Aktiebolag. All rights reserved.<br>
This material is proprietary and may not be reproduced, distributed, or shared without written permission.
</div>